In [48]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

url_locations = "https://1firstcashadvance.org/locations"
urls = []

def parse_info_type1(url):
    count = 0
    response = requests.get(url)
    soup = BeautifulSoup(response.text, "html.parser")
    divs = soup.find_all("div", class_="col")
    address_text, phone_text, hours_text = None, None, None
    for div in divs:
        p_tags = div.find_all('p')
        if len(p_tags) >= 4:
            count += 1
            final_tags = p_tags
            for p in p_tags:
                try:
                    if "Phone:" in p.text:
                        phone_text = p.find('a').text
                    elif "Address:" in p.text:
                        address_text = p.text.replace("Address:", "").strip()
                    elif "Hours:" in p.text:
                        hours_text = p.text.replace("Hours:", "").strip()
                except AttributeError:
                    break

            if url == 'https://1firstcashadvance.org/loans-by-state/payday-loans-colorado/' or url == 'https://1firstcashadvance.org/loans-by-state/payday-loans-oklahoma/':
                response = requests.get(url)
                soup = BeautifulSoup(response.text, "html.parser")
                divs = soup.find_all("div", class_="col")
                address_text, phone_text, hours_text = None, None, None
                count = 0
                for div in divs:
                    p_tags = div.find_all('p')
                    if len(p_tags) >= 4 and count < 2:
                        final_tags = p_tags
                        count += 1
                        print(final_tags)

                list1 = str(final_tags[0]).split('<b>')
                address_text = list1[2].strip("Address:</b> <br/>\n")
                phone_text = list1[3].strip("Phone:</b> <br/>\n")
                hours_text = str(final_tags[1]).strip('<p><b>Hours:</b><br/>')
            
            else:
                address_text = final_tags[1].text.strip('Address: ')
                phone_text = final_tags[2].text.strip('Phone: ')
                hours_text = final_tags[5].text.strip('Hours: ')

            return address_text, phone_text, hours_text
        
    return None, None, None

def parse_links(url_locations):
    response = requests.get(url_locations)
    soup = BeautifulSoup(response.text, "html.parser")
    location = soup.find_all("div", {"class":"col", "style": "text-align: right;"})
    for i in location:
        x = i.find("a")["href"]
        urls.append(x)
    return urls

def parse_info(url):
    response = requests.get(url)
    soup = BeautifulSoup(response.text, "html.parser")
    address = soup.find("p", class_="address")
    phone = soup.find("p", class_="phone")
    hours = soup.find("p", class_="hours")

    address_text = address.text if address else None
    phone_text = phone.text if phone else None
    hours_text = hours.text if hours else None

    if address_text == None or phone_text == None or hours_text == None:
        address_text, phone_text, hours_text = parse_info_type1(url)
    
    return address_text, phone_text, hours_text

q = parse_links(url_locations)
df = pd.DataFrame(columns=["address", "phone", "working hours"])

for x in urls:
    a, b, c = parse_info(x)
    new_row = pd.DataFrame({"link":x, "address": [a], "phone": [b], "working hours": [c]})
    df = pd.concat([df, new_row], ignore_index=True)
df

[<p>The application process for an Oklahoma payday loan is simple and hasn't been long. All you have to do is follow five simple steps:</p>, <p>Complete the application form on our website, then send it.</p>, <p>Wait for a reply. A direct lender needs about 15 minutes to process a loan request and decide if it is possible to lend it. If one of the creditors has a positive response, you will send a direct offer via e-mail.</p>, <p>Read the loan contract. We strongly advise you to read the loan contract carefully. If you have any questions, ask the creditor to help you. Make sure you understand all the terms, the APR (annual percentage rate), and the rates involved.</p>, <p>Sign the contract. If you accept the terms and conditions, the request by signing the loan contract ends.</p>, <p>Receive your money. Once you have your signature, the direct lender will deposit the funds in your bank account. On average, the transfer requires a working day, depending on the score cutting time, among 

,address,phone,working hours,link
0,"2770 Canyon Blvd, Boulder, CO 80302",(720) 428-2247,\nHours:\nMonday – Friday: 8:00 am to 10:00 pm...,https://1firstcashadvance.org/loans-by-state/p...
1,"8008 S Gessner Dr, Houston, TX 77036",(832) 981-1596,\nHours:\nMonday – Sunday: 8:00 AM to 10:00 PM\n,https://1firstcashadvance.org/loans-by-state/p...
2,"7553 Westheimer Rd, Houston, TX 77063",(832) 463-5735,\nHours:\nMonday – Sunday: 8:00 AM to 10:00 PM\n,https://1firstcashadvance.org/locations/7553-w...
3,"5454 North Fwy, Houston, TX 77076",(832) 648-4711,\nHours:\nMonday – Sunday: 8:00 AM to 10:00 PM\n,https://1firstcashadvance.org/locations/5454-n...
4,"1519 Little York Rd, Houston, TX 77093",(832) 648-2604,\nHours:\nMonday – Sunday: 8:00 AM to 10:00 PM\n,https://1firstcashadvance.org/locations/1519-l...
...,...,...,...,...
76,"3639 Ambassador Caffery Pkwy suite 630, Lafaye...",(337) 347-7858,\nHours:\nMonday – Friday: 8:00 am to 10:00 pm...,https://1firstcashadvance.org/loans-by-state/l...
77,"30 River Park Pl W #390, Fresno, CA 93720",(559) 550-6681,\nHours:\nMonday – Friday: 8:00 am to 10:00 pm...,https://1firstcashadvance.org/loans-by-state/p...
78,"6423 University Ave, San Diego, CA 92115",(619) 304-8805,\nHours:\nMonday – Friday: 8:00 am to 10:00 pm...,https://1firstcashadvance.org/loans-by-state/p...
79,"7123 I-30 #46, Little Rock, AR 72209",(501) 406-0133,\nHours:\nMonday – Friday: 8:00 am to 10:00 pm...,https://1firstcashadvance.org/loans-by-state/a...


In [50]:
x = df[df.isnull().any(axis=1)]['link'].tolist()
x

['https://1firstcashadvance.org/loans-by-state/payday-loans-colorado/']

In [40]:
url='https://1firstcashadvance.org/loans-by-state/payday-loans-oklahoma/'
response = requests.get(url)
soup = BeautifulSoup(response.text, "html.parser")
divs = soup.find_all("div", class_="col")
address_text, phone_text, hours_text = None, None, None
count = 0

for div in divs:
    p_tags = div.find_all('p')
    if len(p_tags) >= 4 and count < 2:
        final_tags = p_tags
        count += 1

list1 = str(final_tags[0]).split('<b>')
address_text = list1[2].strip("Address:</b> <br/>\n")
phone_text = list1[3].strip("Phone:</b> <br/>\n")
hours_text = str(final_tags[1]).strip('<p><b>Hours:</b><br/>')
print(address_text, phone_text, hours_text)


1323 SW 59th St, Oklahoma City, OK 73119 (405) 766-7747 
Monday – Friday: 8:00 am to 10:00 pm<br/>
Saturday: 9:00 am to 6:00 pm<br/>
Sunday: Closed


In [26]:
list1 = str(final_tags[0]).split('<b>')
list1


["<p>The application process for an Oklahoma payday loan is simple and hasn't been long. All you have to do is follow five simple steps:</p>"]

In [42]:
url='https://1firstcashadvance.org/loans-by-state/payday-loans-oklahoma/'
response = requests.get(url)
soup = BeautifulSoup(response.text, "html.parser")
divs = soup.find_all("div", class_="col")

for div in divs2:
    p_tags = div.find_all('p')
    if count < 2:
        final_tags = p_tags
        count += 1
        print(final_tags)

list1 = str(final_tags[0]).split('<b>')
address_text = list1[2].strip("Address:</b> <br/>\n")
phone_text = list1[3].strip("Phone:</b> <br/>\n")
hours_text = str(final_tags[1]).strip('<p><b>Hours:</b><br/>')
        

In [43]:
address_text

'1323 SW 59th St, Oklahoma City, OK 73119'